In [1]:
''' material classification '''
import os
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb  # pip install xgboost

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (Linear Kernel)': SVC(kernel='linear'),
    'SVM (RBF Kernel)':    SVC(kernel='rbf'),
    'Random Forest':       RandomForestClassifier(n_estimators=100),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    # 'Gradient Boosting':   GradientBoostingClassifier(),
    # 'Naive Bayes':         GaussianNB(),
    # 'XGBoost':             xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

/home/zhaoy/anaconda3/envs/rwkv-cu121/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import torch
from torchvision import transforms

DATA_DIR = '/data/ssd/zhaoy/datasets/YCB-Slide/real'
SAMPLES_PER_CLASS = 1000
IMG_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # [0,1], shape: (3,H,W)
])

samples = []
for cls_name in os.listdir(DATA_DIR):
    cls_path = Path(DATA_DIR) / cls_name / 'dataset_0' / 'frames'
    if not cls_path.is_dir():
        continue
    for img_file in cls_path.glob("*.jpg"):
        samples.append((str(img_file), cls_name))

samples


[('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0001213.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0003244.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0001027.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0000208.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0000014.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0000420.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0002920.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0000407.jpg',
  '055_baseball'),
 ('/data/ssd/zhaoy/datasets/YCB-Slide/real/055_baseball/dataset_0/frames/frame_0002360.jpg',
  '055_base

In [16]:
from collections import defaultdict
import random

class_to_samples = defaultdict(list)
for img_path, cls in samples:
    class_to_samples[cls].append(img_path)

selected = []
for cls, img_paths in class_to_samples.items():
    if len(img_paths) < SAMPLES_PER_CLASS:
        print(f"Warning: class {cls} only has {len(img_paths)} samples (< {SAMPLES_PER_CLASS})")
        chosen = img_paths
    else:
        chosen = random.sample(img_paths, SAMPLES_PER_CLASS)
    selected.extend([(p, cls) for p in chosen])
    
len(selected)

10000

In [17]:
X, y = [], []
for img_path, cls in selected:
    try:
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform(img)   # (3,64,64)
        feat = img_tensor.view(-1).numpy()  # flatten 成 (3*64*64,)
        X.append(feat)
        y.append(cls)
    except Exception as e:
        print(f"Failed to load {img_path}: {e}")

X = np.array(X)
y = np.array(y)
print("Feature matrix:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

Feature matrix: (10000, 12288)


In [18]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

results = {}
for name, clf in classifiers.items():
    print(f"\n===== Training {name} =====")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # 1) 基本指标
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    results[name] = {"accuracy": acc, "balanced_acc": bal_acc}

    print(f"{name} Accuracy: {acc:.4f}")
    print(f"{name} Balanced Accuracy: {bal_acc:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))


===== Training Logistic Regression =====
Logistic Regression Accuracy: 0.9918
Logistic Regression Balanced Accuracy: 0.9918
Classification Report:
                        precision    recall  f1-score   support

        004_sugar_box       1.00      1.00      1.00       400
  005_tomato_soup_can       0.99      0.99      0.99       400
   006_mustard_bottle       0.97      0.99      0.98       400
  021_bleach_cleanser       0.98      0.97      0.98       400
              025_mug       0.99      1.00      1.00       400
      035_power_drill       0.99      0.98      0.98       400
         037_scissors       1.00      1.00      1.00       400
042_adjustable_wrench       0.99      0.98      0.99       400
           048_hammer       1.00      1.00      1.00       400
         055_baseball       1.00      1.00      1.00       400

             accuracy                           0.99      4000
            macro avg       0.99      0.99      0.99      4000
         weighted avg       0.